In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, str(Path.cwd().parent))
from dataloader import hed_jitter_batch
from train import make_region_idx

In [ ]:
GRAY = 0.55


def visualize_masking(img, side=12, k=84, decay=0.0, grid=16, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    img = np.asarray(img, dtype=np.float32)
    if img.max() > 1.0:
        img = img / 255.0
    patch = img.shape[0] // grid

    keep, target = make_region_idx(1, grid, torch.device("cpu"), side, k, decay)
    visible = np.zeros(grid * grid, dtype=bool)
    visible[keep[0].numpy()] = True
    targeted = np.zeros(grid * grid, dtype=bool)
    targeted[target[0].numpy()] = True

    up = lambda m: np.kron(m.reshape(grid, grid), np.ones((patch, patch), dtype=bool))[..., None]
    vis, tgt = up(visible), up(targeted)
    gray = np.full_like(img, GRAY)

    left = np.where(vis, img, gray)
    right = np.where(vis, img, np.where(tgt, 0.5 * img + 0.5, gray))

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
    for ax, panel, title in zip(axes, (left, right), (f"context  side={side}  {side ** 2}/{grid * grid} patches", f"targets  k={k}  decay={decay}")):
        ax.imshow(np.clip(panel, 0, 1))
        ax.set_xticks([]), ax.set_yticks([])
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    return fig

In [ ]:
img = plt.imread("sample.png") if Path("sample.png").exists() else np.random.default_rng(0).random((224, 224, 3))

visualize_masking(img, side=8, k=84, decay=0.5, seed=0)
plt.show()

In [ ]:
def jitter_sweep(img, sigmas=(0.02, 0.02, 0.02, 0.02, 0.02), seed=0):
    img = np.asarray(img, dtype=np.float32)
    if img.max() > 1.0:
        img = img / 255.0
    x = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1)[None]

    fig, axes = plt.subplots(1, len(sigmas), figsize=(2.6 * len(sigmas), 3.2))
    for ax, s in zip(np.atleast_1d(axes), sigmas):
        # torch.manual_seed(seed)
        out = hed_jitter_batch(x, s)[0].permute(1, 2, 0).numpy()
        ax.imshow(np.clip(out, 0, 1))
        ax.set_xticks([]), ax.set_yticks([])
        ax.set_title(f"sigma={s}   mad={np.abs(out - img).mean():.3f}", fontsize=10)
    fig.tight_layout()
    return fig


jitter_sweep(img)
plt.show()